# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenken24/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Freestyle — Content Cannibalization Detection.**

**Question:** which pages of the *same client* compete for the *same search queries*, so that the clicks get split between them?

**Picture it:** two shops of the same chain open on one street. Customers (clicks) get divided, and neither shop looks busy. The fix is a decision (close one, or make them sell different things), not a new shop.

**Why freestyle, not one of the four lanes:** Lane 2 (ranked review queue with reason codes) is my closest lane and my **fallback**. Lane 3 lists "cannibalization-risk pages" as a cluster name, but clustering page-level metrics cannot show *shared queries*. This question needs query-level data (which page showed up for which query, with impressions and clicks), and the warehouse table `fact_content_query_90d` is built for exactly that (per the data guide: one row per client x content x query hash).

**Plan (one paragraph):** Week 3 - write the data contract on the query table (client x content x query hash). Week 4 - signal audit plus a rule baseline ("pages share at least N queries with at least M impressions each"), then **confirm this lane or fall back to Lane 2**. Weeks 5-6 - try a better score only if it beats the baseline; validate with a hand-checked top 20 and a stability check. Week 7 - ranked action list with reason codes and the write-up.

**Key rule I start from:** similar titles (or similar metadata) alone do not prove cannibalization. Query-level evidence is much stronger.

In [1]:
import os, sys, subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zenken24/flyrank"
REPO_DIR = "flyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

CSV = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV), "starter CSV not found - are you at the repo root?"
df = pd.read_csv(CSV)

print(f"{len(df):,} pages x {df.shape[1]} columns, {df.client_id.nunique()} clients")
# Does the starter data have anything query-level / URL / title / text?
print("query/url/title/text columns:", [x for x in df.columns if any(k in x.lower() for k in ("query", "url", "title", "text"))])

30,000 pages x 44 columns, 32 clients
query/url/title/text columns: []


## 2. The question: decision, action, cost of a wrong call

**Search question:** which pages of the same client compete for the same queries, enough to split their clicks, so an editor should look at them first?

**Decision it improves:** *which group of pages an editor reviews first.* (Not "predict cannibalization" - that is not a decision yet.)

**Unit of analysis:** a **page group** = pages of one client that all get impressions for the same query.
Example: pages A, B, C all show up for query `q_17` (a hashed query). A gets 60% of the impressions, B 30%, C 10%. That is **one group of 3 pages**, and it is one row of my final list.

**Output:** a ranked review list of groups. Each row carries reason codes a human can check, for example *shares 5 queries*, *top page holds only 55% of impressions*, *both pages sit on page 1-2*, plus a suggested review action.

**Action (who does what):** a content editor opens the top groups and picks one:
- **merge / redirect** the weaker page into the stronger one,
- **differentiate** - re-focus one page on a different intent,
- **fix internal links** so one page is clearly the main one,
- **leave alone / monitor** if the pages serve different readers.

**Cost of a wrong call**

| Mistake | What happens | Cost |
|---|---|---|
| **False alarm** (group flagged, pages are not really competing) | Editor spends hours; worse, merges two pages that served different readers and loses traffic | Wasted time **plus** possible lost traffic, hard to undo |
| **Miss** (real cannibalization not flagged) | Clicks stay split, pages stay stuck | A missed gain only - the status quo |

A false alarm costs more than a miss, so the list is **"review first", never "auto-merge"**.

**Task type:** ranking ("which ones first?"). Nobody has written down which groups are *truly* cannibalizing, so there is **no observed label** - v1 is a transparent evidence score, not a trained classifier. Metrics named *before* building: **hand-checked precision@20** (of my top 20 groups, how many pass a written evidence checklist) and **stability** (a flagged group is still flagged when I compare an earlier and a later part of the window).

**Why data (and maybe ML) helps at all:** a plain rule like "same keyword = duplicate" is wrong in both directions: it flags pages that serve different readers (section 3 shows this on real numbers), and it can miss pages that compete on *related* queries that do not match exactly. The real signal is query overlap weighted by impressions, position and how clicks split, spread over millions of rows nobody can eyeball (the query table has about 2.4M rows, per the data guide). I start with a simple rule; ML only earns a place if it beats that rule on my checklist.

**One-paragraph frame:** For content editors deciding which same-client page groups to review first, I will build a ranked list with reason codes from query-level search data (impressions, clicks and position per page per query), scoring the evidence that pages split the same queries, measured by hand-checked precision@20 and stability across the window. A wrong call costs editor time and possibly merged pages that served different readers; a keyword-match rule is not enough because matching metadata is not evidence of competing. I will claim only observed overlap and decision-support rankings.

In [2]:
# Grain probe: in the starter CSV, one row = one PAGE (content_id must be unique).
print("one row per page:", df.content_id.is_unique)
# So a real 'page group' (pages sharing a query) cannot be built from this file.
# That needs fact_content_query_90d (client x content x query hash) in Week 3.

one row per page: True


## 3. Quick look at the data (2-3 real numbers)

The starter CSV has **one row per page and no query column** (the cell above prints an empty list). So I cannot see real query overlap yet.

What I *can* do is a rough **proxy**: treat pages of the same client that share the same *keyword fingerprint* (search volume + competition + CPC) as **candidates**, then ask if the pool is big enough to be worth 7 weeks. A fingerprint is a hint, **not** proof.

In [3]:
KEYS = ["client_id", "search_volume", "competition", "cpc"]   # keyword fingerprint (proxy)
kw = df.dropna(subset=["search_volume", "competition", "cpc"])        # pages that have keyword data

# --- Loose fingerprint: any page that shares the fingerprint with another page of the same client ---
loose_size = kw.groupby(KEYS).content_id.transform("size")
loose_pages = int((loose_size > 1).sum())
loose_share = loose_pages / len(kw)
loose_largest = int(loose_size.max())
zero_volume_pages = int((kw.search_volume == 0).sum())

# --- Specific fingerprint: volume, competition AND cpc all > 0 (less likely to match by accident) ---
spec = kw[(kw.search_volume > 0) & (kw.competition > 0) & (kw.cpc > 0)]
size = spec.groupby(KEYS).content_id.transform("size")
cand = spec[size > 1]                                                  # pages that share a fingerprint

g = cand.groupby(KEYS).agg(pages=("content_id", "size"),
                           imp=("impressions_90d", "sum"),
                           top=("impressions_90d", "max"),
                           n100=("impressions_90d", lambda s: int((s >= 100).sum())))
live = g[g.n100 >= 2].copy()                    # >=2 pages with >=100 impressions: enough volume to matter
live["top_share"] = live.top / live.imp         # how much of the group's impressions the top page holds

n_groups = len(live)
n_pages = int(live.pages.sum())
n_clients = live.reset_index().client_id.nunique()
n_split = int((live.top_share < 0.8).sum())
median_top = float(live.top_share.median())
non_top_imp = int((live.imp - live.top).sum())
non_top_share = non_top_imp / df.impressions_90d.sum()

print("NUMBER 1 - candidate groups (specific fingerprint, >=2 pages with >=100 impressions)")
print(f"  {n_groups} groups, {n_pages} pages, {n_clients} of {df.client_id.nunique()} clients")
print("NUMBER 2 - how impressions are spread inside those groups")
print(f"  top page holds <80% of the group's impressions in {n_split} of {n_groups} groups ({n_split/n_groups:.0%}); median top-page share = {median_top:.0%}")
print("WARNING NUMBER - the loose fingerprint")
print(f"  {loose_pages:,} pages ({loose_share:.0%} of pages with keyword data) 'share' a fingerprint; largest group = {loose_largest:,} pages; {zero_volume_pages:,} pages have search volume 0")
print("CAUTION NUMBER - size of the prize on this crude proxy")
print(f"  pages below the top page hold {non_top_imp:,} impressions = {non_top_share:.2%} of all impressions in the slice")

NUMBER 1 - candidate groups (specific fingerprint, >=2 pages with >=100 impressions)
  160 groups, 345 pages, 13 of 32 clients
NUMBER 2 - how impressions are spread inside those groups
  top page holds <80% of the group's impressions in 101 of 160 groups (63%); median top-page share = 73%
WARNING NUMBER - the loose fingerprint
  20,073 pages (73% of pages with keyword data) 'share' a fingerprint; largest group = 3,066 pages; 11,081 pages have search volume 0
CAUTION NUMBER - size of the prize on this crude proxy
  pages below the top page hold 374,810 impressions = 0.24% of all impressions in the slice


**What these numbers say (simple words)**

1. **The pool is real and not one client's quirk:** 160 candidate groups (345 pages) across 13 of 32 clients, each with at least 2 pages that have 100+ impressions in 90 days.
2. **Impressions are often spread, not won by one page:** in 101 of the 160 groups (63%) the top page holds under 80% of the group's impressions (median 73%). *Directional only - impressions are not clicks lost.*
3. **The loose fingerprint fails, and that is the lesson:** 20,073 pages (73% of pages with keyword data) "share" a fingerprint and the biggest group has 3,066 pages. That is a collision (for example 11,081 pages have search volume 0), not cannibalization. Same idea as "similar titles do not prove it": **matching metadata is not evidence; query-level overlap is.**
4. **Honest caution:** pages below the top page hold only about 0.24% of all impressions in this slice. On this crude proxy the prize looks small. I do **not** treat that as the real size (the proxy only sees exact fingerprint matches), but it is a risk. **Kill rule:** if, on the real query table in Week 3-4, I cannot find a meaningful number of groups that share real queries with split impressions, I switch to Lane 2 before the end of Week 4.

## 4. Careful words: what I can and can't claim

**What my work will be able to say**

- **Observed:** "These pages of one client received impressions for the same queries in this 90-day window; the top page held 62% of them."
- **Directional:** "Impressions are split across the group, which suggests the pages may be competing."
- **Decision-support:** "These 20 groups are the best ones to review first, because of these reason codes."

**What it will never say**

- ✗ "Cannibalization *caused* this page's decline" - I only observe; I run no experiment.
- ✗ "Merging these pages will recover the traffic" - no causal design, so no promise.
- ✗ "Google picked the wrong page" or "Google penalises duplicates" - I have no view into Google's algorithm.
- ✗ "These two pages are duplicate content" - the data has no titles, URLs or article text, and similar metadata is not similar content.
- ✗ "Same keyword fingerprint = same keyword" - it is only a proxy (section 3).

**Example of careful wording:** not *"Page A steals clicks from page B"*, but *"A and B share 5 queries; A holds 62% of the impressions."*

No client names, domains, URLs or raw queries appear in this notebook - only pseudonymized ids and aggregate counts.

In [4]:
# Check: every number I quote in the markdown above equals what the data gives.
assert (n_groups, n_pages, n_clients) == (160, 345, 13)
assert (n_split, round(median_top, 2)) == (101, 0.73)
assert (loose_pages, loose_largest, zero_volume_pages) == (20073, 3066, 11081)
assert round(100 * loose_share) == 73 and round(100 * non_top_share, 2) == 0.24
print("OK - the numbers in the text match the data.")

OK - the numbers in the text match the data.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.